In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

chat_model = ChatOpenAI(
    model="local-llm",
    base_url="http://localhost:8080/v1",
    temperature=0.7
)


# create subagents

subagent_1 = create_agent(
    model= chat_model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=chat_model,
    tools=[square]
)

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=chat_model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='1af1e939-2930-48f0-ada3-c81ec6c57264'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 157, 'total_tokens': 297, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e2b-it.gguf', 'system_fingerprint': 'b11429-d81235049', 'id': 'chatcmpl-vQ5dkY2AG6p7jRURLZNrglG1pk3x4sIv', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1638a-459c-76b3-aea3-e11967ced03d-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'athvEGEgcMhTYmfQMeaD2MUOnvP9xNrJ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 140, 'total_tokens

Lets level up our example for multi agent.

- Let have two seperate server running models,one process image and other process text
```bash
    llama-server -m ./models/gemma4-e2b-it.gguf --port 8080 -c 128000 -ngl 99
```
- Server which can process both image and text
```bash
    llama-server -m ./models/gemma4-e2b-it.gguf --mmproj ./models/gemma4-e2b-it-mmproj.gguf --port 8081 -c 4096 -ngl 99
```

In [20]:


text_llm = ChatOpenAI(
    model="local-text-process",
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
    temperature=0.7,
)

image_llm = ChatOpenAI(
    model = 'local-image-process',
    base_url="http://localhost:8081/v1",
    temperature=0.7
)

image_processing_agent = create_agent(
    model=image_llm,
)

# create subagents
subagent_1 = create_agent(
    model= text_llm,
)
subagent_2 = create_agent(
    model=image_llm
)

In [24]:
from pathlib import Path
import base64
@tool
def call_subagent_1(text_input: str):
    """Call Subagent 1 to handle text reasoning, logic, and question answering."""
    return subagent_1.invoke({
        'messages': [HumanMessage(content=text_input)]
    })

@tool
def call_subagent_2(image_path_or_b64: str, prompt: str = "Analyze this image."):
    """Call Subagent 2 to process and describe images (Port 8081).
    Accepts either a file path to an image or a base64-encoded string.
    """
    path = Path(image_path_or_b64)
    if path.is_file():
        with open(path, "rb") as img_file:
            img_b64 = base64.b64encode(img_file.read()).decode("utf-8")
    else:
        img_b64 = image_path_or_b64


    multimodal_question = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image", "base64": img_b64, "mime_type": "image/png"}
    ])

    return subagent_2.invoke(
        {"messages": [multimodal_question]}
    )

In [26]:
main_agent = create_agent(
    model=chat_model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt=(
        "You are a helpful assistant who can call subagents to answer user input. "
        "Use call_subagent_1 for text questions and reasoning. "
        "Use call_subagent_2 when the query involves image analysis or image files."
    ),
)

In [27]:
question = HumanMessage(
    """from eOpenAIInvalidRequestError: Error code: 400 - {'error': {'code': 400, 'message': "model 'local-image-process' not found", 'type': 'invalid_request_error'}}During task with name 'model' and id 'e5c7ed4a-6ca6-2802-14c6-5579797edb2c'  and screenshot related to it at /Users/manikanta.kodimala/Traning/Learning-AI/module-1/resources/image.png"""
)

response = main_agent.invoke({
    "messages": [question]
})

In [34]:
pprint(response["messages"])

[HumanMessage(content='from eOpenAIInvalidRequestError: Error code: 400 - {\'error\': {\'code\': 400, \'message\': "model \'local-image-process\' not found", \'type\': \'invalid_request_error\'}}During task with name \'model\' and id \'e5c7ed4a-6ca6-2802-14c6-5579797edb2c\'  and screenshot related to it at /Users/manikanta.kodimala/Traning/Learning-AI/module-1/resources/image.png', additional_kwargs={}, response_metadata={}, id='2d7ce84b-09cc-43b2-a84a-5d523baaf810'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 795, 'prompt_tokens': 351, 'total_tokens': 1146, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e2b-it.gguf', 'system_fingerprint': 'b11429-d81235049', 'id': 'chatcmpl-m4fICrpMlYBHakoljctAMgvGmHRj0834', 'finish_reason': 'to

### second trial changing system prompt

In [35]:
main_agent = create_agent(
    model=chat_model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt=(
        "You are an orchestration agent responsible for routing user queries to subagents.\n\n"
        "Instructions:\n"
        "1. Analyze the user prompt carefully.\n"
        "2. Check if the user prompt mentions an image file path (e.g., paths ending in .png, .jpg, or file system paths) or image data.\n"
        "3. If an image file path is present:\n"
        "   - Extract the exact file path.\n"
        "   - Call `call_subagent_2` with `image_path_or_b64` set to the extracted path, and pass the rest of the text query as the `prompt` parameter.\n"
        "4. If no image path or image file is present, route the query to `call_subagent_1`."
    ),
)

In [36]:
question = HumanMessage(
    """from eOpenAIInvalidRequestError: Error code: 400 - {'error': {'code': 400, 'message': "model 'local-image-process' not found", 'type': 'invalid_request_error'}}During task with name 'model' and id 'e5c7ed4a-6ca6-2802-14c6-5579797edb2c'  and screenshot related to it at /Users/manikanta.kodimala/Traning/Learning-AI/module-1/resources/image.png"""
)

response = main_agent.invoke({
    "messages": [question]
})

In [38]:
pprint(response["messages"][1].tool_calls)


[{'args': {'image_path_or_b64': '/Users/manikanta.kodimala/Traning/Learning-AI/module-1/resources/image.png',
           'prompt': 'from eOpenAIInvalidRequestError: Error code: 400 - '
                     '{\'error\': {\'code\': 400, \'message\': "model '
                     '\'local-image-process\' not found", \'type\': '
                     "'invalid_request_error'}}During task with name 'model' "
                     "and id 'e5c7ed4a-6ca6-2802-14c6-5579797edb2c'"},
  'id': '4cVeeFRMDe7ACfFkwY2dW0KLCcu2ZeJU',
  'name': 'call_subagent_2',
  'type': 'tool_call'}]


#### Third Trial by changing the system prompt

In [39]:
main_agent = create_agent(
    model=chat_model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt=(
        "You are an orchestration agent responsible for routing user queries.\n\n"
        "Instructions:\n"
        "1. If a query contains both a text question/error and an image path:\n"
        "   - Call `call_subagent_1` with the text error log for code analysis.\n"
        "   - Call `call_subagent_2` with the image path to analyze the visual content.\n"
        "2. Combine the results from both subagents into a final answer."
    ),
)

In [40]:
question = HumanMessage(
    """from eOpenAIInvalidRequestError: Error code: 400 - {'error': {'code': 400, 'message': "model 'local-image-process' not found", 'type': 'invalid_request_error'}}During task with name 'model' and id 'e5c7ed4a-6ca6-2802-14c6-5579797edb2c'  and screenshot related to it at /Users/manikanta.kodimala/Traning/Learning-AI/module-1/resources/image.png"""
)

response = main_agent.invoke({
    "messages": [question]
})

In [46]:
pprint(response["messages"][1].tool_calls)

print("+++++++++++++++++++++++++++++\n")

pprint(response["messages"][1].content)

print("+++++++++++++++++++++++++++++\n")

pprint(response["messages"][2].content)


[{'args': {'text_input': 'from eOpenAIInvalidRequestError: Error code: 400 - '
                         '{\'error\': {\'code\': 400, \'message\': "model '
                         '\'local-image-process\' not found", \'type\': '
                         "'invalid_request_error'}}During task with name "
                         "'model' and id "
                         "'e5c7ed4a-6ca6-2802-14c6-5579797edb2c'"},
  'id': '1JbSnjEjkm6HtOW4ncD5fSNqU2KeK8Jz',
  'name': 'call_subagent_1',
  'type': 'tool_call'},
 {'args': {'image_path_or_b64': '/Users/manikanta.kodimala/Traning/Learning-AI/module-1/resources/image.png'},
  'id': '7AjO98VTYGGbcrmVo3tUtCyjCCZtMD4c',
  'name': 'call_subagent_2',
  'type': 'tool_call'}]
+++++++++++++++++++++++++++++

''
+++++++++++++++++++++++++++++

("{'messages': [HumanMessage(content='from eOpenAIInvalidRequestError: Error "
 'code: 400 - {\\\'error\\\': {\\\'code\\\': 400, \\\'message\\\': "model '
 '\\\'local-image-process\\\' not found", \\\'type\\\': '
 "